# 03 -- Evaluate (Step 2: eval harness + the Stockfish end of the reference set)

Plan v3, Step 2. Every later model is rated against a **frozen reference set**. One rating is pinned by
definition -- Stockfish `UCI_Elo` 1320 = 1320 -- and every other member is measured once, then frozen.
Only the Stockfish end can be measured now: random and greedy lose every game to Stockfish 1320, so their
ratings wait for the Step 5 checkpoints to bridge the gap.

**Gate:**
1. Smoke tests pass: random loses almost every game to Stockfish depth 1, and an agent that keeps playing
   illegal moves forfeits on its third attempt.
2. Stockfish 1500 and 1700 are rated (1320 pinned) with sensible confidence intervals, then frozen to Drive.
3. Game timings are recorded (needed to size eval in Step 6).

Runs on CPU -- no GPU runtime needed.

In [ ]:
# Setup: clone the repo, install deps, mount Drive, fetch the pinned Stockfish.
import os
REPO_URL = "https://github.com/derekhe99/Chess_Bot"
REPO_DIR = "/content/Chess_Bot"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/chess-ai'
RESULTS = os.path.join(DRIVE_ROOT, 'results')
os.makedirs(RESULTS, exist_ok=True)

from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
# Time-limited Stockfish plays stronger on a faster CPU, so the frozen ratings are tied to this hardware -- log it.
CPU_MODEL = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
print("CPU:", CPU_MODEL, "| cores:", os.cpu_count())

## Parameters

All the knobs for this run (they move to `configs/eval.yaml` once the config design is agreed).
The game counts follow the plan's adaptive rule: 20 games per matchup to start, then 20 more only in
matchups where a rating isn't tight enough yet (95% CI wider than +/-100, or a score outside 20-80%),
up to a cap.

In [ ]:
import json
import time
from dataclasses import asdict

from eval.anchors import (REFERENCE_SET, SMOKE_TEST_OPPONENT, FrozenMember, FrozenRegistry,
                          build_agents, close_agents, fixed_ratings)
from eval.elo import format_table
from eval.match import play_game, play_match, rate_adaptively, summarize

N_SMOKE = 10              # random vs Stockfish depth-1 games
OPENING_PLIES = 4         # random opening half-moves, shared by each color-swapped pair of games
N_BOOT = 1000             # bootstrap resamples for the confidence intervals
START_GAMES = 20          # adaptive rule: first batch per matchup
STEP_GAMES = 20           #   games added to a matchup that still needs more
MAX_GAMES_PER_PAIR = 100  #   cap per matchup
CI_HALF_WIDTH = 100       #   target: 95% CI no wider than +/- this
SCORE_BAND = (0.2, 0.8)   #   scores outside this band count as lopsided
REGISTRY_PATH = os.path.join(RESULTS, "reference_set.json")

SF_NAMES = [s.name for s in REFERENCE_SET if s.kind == "stockfish"]   # sf_1320, sf_1500, sf_1700
PINNED = fixed_ratings(REFERENCE_SET)                                   # {"sf_1320": 1320.0}
CANDIDATES = [n for n in SF_NAMES if n not in PINNED]                   # rated here, then frozen
PAIRS = [("sf_1320", "sf_1500"), ("sf_1500", "sf_1700"), ("sf_1320", "sf_1700")]

for s in list(REFERENCE_SET) + [SMOKE_TEST_OPPONENT]:
    role = "pinned" if s.name in PINNED else "rate now" if s.name in CANDIDATES else \
           "smoke test only" if s is SMOKE_TEST_OPPONENT else "rated in Step 5"
    print(f"{s.name:<10} elo={str(s.elo):<5} limit={str(s.limit):<14} {role}")
agents = build_agents(list(REFERENCE_SET) + [SMOKE_TEST_OPPONENT])

## Smoke test 1 -- random vs Stockfish depth 1
Random should lose nearly every game. If it doesn't, something in the game runner is broken.

In [ ]:
recs_smoke = play_match(agents["random"], agents["sf_depth1"], N_SMOKE, opening_plies=OPENING_PLIES)
s = summarize(recs_smoke, "random")
print(f"random: {s['wins']}W {s['draws']}D {s['losses']}L (score {s['score']:.2f})")
SMOKE_RANDOM_LOSES = s["score"] <= 0.1

## Smoke test 2 -- three strikes
An agent that only ever plays illegal moves must forfeit on its third attempt, before any move is made
(plan v3, Sec 0 "Illegal moves"). This is the runner's default -- nothing is passed in.

In [ ]:
class AlwaysIllegal:
    """Stands in for a confused unmasked LLM: every answer is impossible."""
    name = "always_illegal"
    def select_move(self, board):
        return "a1a1"

g = play_game(AlwaysIllegal(), agents["random"])
print(g.termination, "| illegal attempts:", g.illegal_attempts["white"], "| plies:", g.plies)
SMOKE_THREE_STRIKES = g.termination == "illegal_move" and g.illegal_attempts["white"] == 3 and g.plies == 0

## Rate Stockfish 1500 and 1700 (1320 pinned)
All three Stockfish levels play each other; only 1320's rating is fixed. Expect 1320 < 1500 < 1700. The
measured gaps may be smaller or larger than the nominal 180 / 200: Stockfish calibrated `UCI_Elo` at a much
longer time control (and states +/-100 Elo), and we play at 0.1 s per move -- a gap that differs from
nominal is a finding to note, not a failure.

Timing on a 2-core test machine: ~8 s per game, so the first batch (60 games) takes ~8 min, and the worst case
(every matchup hits the 100-game cap, 300 games) ~40 min. Colab's CPU may differ -- the Timing cell below records it.

In [ ]:
t0 = time.perf_counter()
est, recs_sf, log = rate_adaptively(
    agents, PAIRS, PINNED, CANDIDATES,
    start_games=START_GAMES, step_games=STEP_GAMES, max_games_per_pair=MAX_GAMES_PER_PAIR,
    ci_half_width=CI_HALF_WIDTH, score_band=SCORE_BAND, n_boot=N_BOOT, opening_plies=OPENING_PLIES)
elapsed = time.perf_counter() - t0
print()
print(format_table(est.values()))
print("\nGames per matchup:", log[-1]["games"])
if log[-1]["capped"]:
    print("Hit the cap without meeting the rule (judge by hand):", log[-1]["capped"])

## Timing
How long games take on this CPU. Step 6 uses this to size evaluation.

In [ ]:
print(f"{len(recs_sf)} Stockfish games in {elapsed / 60:.1f} min -> {elapsed / len(recs_sf):.1f} s per game")
for n in SF_NAMES:
    s = summarize(recs_sf, n)
    print(f"{n:<8} {s['avg_plies']:.0f} plies/game, {s['avg_move_seconds']:.3f} s/move")

## Gate check

In [ ]:
r = {n: e.rating for n, e in est.items()}
checks = {
    "smoke: random loses to Stockfish depth 1": SMOKE_RANDOM_LOSES,
    "smoke: third illegal attempt forfeits": SMOKE_THREE_STRIKES,
    "sf_1500 and sf_1700 are on the scale (not flagged)": all(est[n].at_bound is None for n in CANDIDATES),
    "ordered: 1320 < sf_1500 < sf_1700": 1320 < r["sf_1500"] < r["sf_1700"],
    f"95% CIs within +/-{CI_HALF_WIDTH}": all((est[n].ci_high - est[n].ci_low) / 2 <= CI_HALF_WIDTH
                                               for n in CANDIDATES),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
GATE = all(checks.values())
print("\nGATE", "PASSED" if GATE else "NOT PASSED -- don't freeze; look at the failing line")

## Freeze
Only if the gate passed. Adds sf_1500 and sf_1700 to the reference-set registry on Drive. A member that's
already frozen is skipped, never re-rated -- to redo it, discuss first, then delete the file deliberately.

In [ ]:
reg = FrozenRegistry(REGISTRY_PATH)
if not GATE:
    print("NOT freezing -- the gate didn't pass.")
else:
    specs = {s.name: s for s in REFERENCE_SET}
    for n in CANDIDATES:
        if n in reg:
            print(f"{n} already frozen at {reg.ratings()[n]:.0f} -- skipped (never re-rated)")
            continue
        e = est[n]
        reg.add(FrozenMember(n, e.rating, e.ci_low, e.ci_high, e.games, spec=asdict(specs[n]),
                             notes={"step": 2, "pinned": PINNED, "games_per_matchup": log[-1]["games"],
                                    "cpu": CPU_MODEL, "cpu_cores": os.cpu_count()}))
    reg.save()
    print("Frozen reference set:", {n: round(v) for n, v in reg.ratings().items()}, "->", REGISTRY_PATH)

In [ ]:
# Keep every game (PGN) and the adaptive-rule log on Drive, then shut the engines down.
with open(os.path.join(RESULTS, "step2_games.pgn"), "w") as f:
    for rec in recs_smoke + [g] + recs_sf:
        f.write(rec.pgn + "\n\n")
with open(os.path.join(RESULTS, "step2_rating_log.json"), "w") as f:
    json.dump(log, f, indent=2)
close_agents(agents)
print("saved", len(recs_smoke) + 1 + len(recs_sf), "games")